# Simulated Retail Sales Generator

This notebook creates fake but realistic retail transaction data for the retail sales dashboard project.

- 5 NZ stores, 10 products
- 30 days of backfilled history
- An optional live-streaming mode to insert new transactions continuously



In [ ]:
import argparse
import random
import sqlite3
import time
from datetime import datetime, timedelta

DB_PATH = "retail.db"

## Store and product data

In [ ]:
# I picked NZ store locations so the project feels local
STORES = [
    (1, "Newmarket", "Auckland", 1.3),
    (2, "Sylvia Park", "Auckland", 1.5),
    (3, "Botany", "Auckland", 1.1),
    (4, "Albany", "Auckland", 1.0),
    (5, "Riccarton", "Christchurch", 0.9),
]

# product_id, name, category, unit price (NZD), popularity weight
PRODUCTS = [
    (1, "Electric Shaver", "Grooming", 149.00, 3),
    (2, "Beard Trimmer", "Grooming", 79.00, 5),
    (3, "Hair Clipper", "Grooming", 99.00, 4),
    (4, "Hair Dryer", "Hair Care", 129.00, 4),
    (5, "Straightener", "Hair Care", 159.00, 3),
    (6, "Electric Toothbrush", "Oral Care", 119.00, 4),
    (7, "Replacement Blades", "Accessories", 29.00, 9),
    (8, "Shaving Foam", "Accessories", 9.50, 10),
    (9, "Travel Case", "Accessories", 24.00, 5),
    (10, "Gift Set", "Gifts", 89.00, 2),
]

# I weight hours so that lunchtime and after-work are busier
HOUR_WEIGHTS = {9: 3, 10: 5, 11: 6, 12: 9, 13: 8, 14: 6, 15: 6, 16: 8, 17: 9, 18: 5, 19: 2}

STORE_HOURS = range(9, 20)

## Database setup

In [ ]:
def connect():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA journal_mode=WAL")  # lets a dashboard read while I write
    return conn


def init_db(conn):
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS stores (
            store_id INTEGER PRIMARY KEY,
            store_name TEXT,
            city TEXT
        );
        CREATE TABLE IF NOT EXISTS products (
            product_id INTEGER PRIMARY KEY,
            product_name TEXT,
            category TEXT,
            unit_price REAL
        );
        CREATE TABLE IF NOT EXISTS transactions (
            transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
            transaction_time TEXT,
            store_id INTEGER,
            product_id INTEGER,
            quantity INTEGER,
            unit_price REAL,
            revenue REAL,
            FOREIGN KEY (store_id) REFERENCES stores(store_id),
            FOREIGN KEY (product_id) REFERENCES products(product_id)
        );
        CREATE INDEX IF NOT EXISTS idx_tx_time ON transactions(transaction_time);
        """
    )
    conn.executemany(
        "INSERT OR IGNORE INTO stores VALUES (?, ?, ?)", [(s[0], s[1], s[2]) for s in STORES]
    )
    conn.executemany(
        "INSERT OR IGNORE INTO products VALUES (?, ?, ?, ?)", [p[:4] for p in PRODUCTS]
    )
    conn.commit()

## Transaction generation logic

In [ ]:
def make_transaction(ts, outage_store=None):
    """I build one transaction row for a given timestamp."""
    weights = [s[3] for s in STORES]
    store = random.choices(STORES, weights=weights)[0]

    # I simulate an outage or slow day at one store so the alerts have something to catch
    if outage_store is not None and store[0] == outage_store and random.random() < 0.8:
        return None

    product = random.choices(PRODUCTS, weights=[p[4] for p in PRODUCTS])[0]
    quantity = random.choices([1, 2, 3], weights=[80, 15, 5])[0]

    # Small price variation to mimic discounts
    price = round(product[3] * random.choice([1.0, 1.0, 1.0, 0.9, 0.85]), 2)
    revenue = round(price * quantity, 2)
    return (ts.strftime("%Y-%m-%d %H:%M:%S"), store[0], product[0], quantity, price, revenue)

## Backfill: generate 30 days of history

In [ ]:
def backfill(conn, days):
    """I generate historical data so the charts have trends from day one."""
    rows = []
    start = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0) - timedelta(days=days)
    for d in range(days):
        day = start + timedelta(days=d)
        weekend_boost = 1.4 if day.weekday() >= 5 else 1.0
        for hour, w in HOUR_WEIGHTS.items():
            n = int(w * 6 * weekend_boost * random.uniform(0.8, 1.2))
            for _ in range(n):
                ts = day.replace(hour=hour, minute=random.randint(0, 59), second=random.randint(0, 59))
                tx = make_transaction(ts)
                if tx:
                    rows.append(tx)
    conn.executemany(
        "INSERT INTO transactions (transaction_time, store_id, product_id, quantity, unit_price, revenue) "
        "VALUES (?, ?, ?, ?, ?, ?)",
        rows,
    )
    conn.commit()
    print(f"Backfilled {len(rows)} transactions over {days} days")

## Live streaming: insert new transactions continuously

This runs forever until stopped, so only run this cell when you actually want live data flowing (for example, while Power BI is open in DirectQuery mode). Interrupt the kernel to stop it.

In [ ]:
def stream(conn, interval, outage_store=None):
    """I insert new transactions continuously to imitate live sales."""
    print(f"Streaming transactions every ~{interval}s. Interrupt the kernel to stop.")
    count = 0
    try:
        while True:
            now = datetime.now()
            tx = make_transaction(now, outage_store)
            if tx:
                conn.execute(
                    "INSERT INTO transactions (transaction_time, store_id, product_id, quantity, unit_price, revenue) "
                    "VALUES (?, ?, ?, ?, ?, ?)",
                    tx,
                )
                conn.commit()
                count += 1
                if count % 10 == 0:
                    print(f"{count} live transactions inserted")
            time.sleep(random.uniform(0.3, 1.5) * interval)
    except KeyboardInterrupt:
        print(f"Stopped after {count} live transactions")

## Run it

Connect to the database, create the tables, and generate 30 days of history.

In [ ]:
conn = connect()
init_db(conn)
backfill(conn, days=30)

Backfilled 13362 transactions over 30 days


### Optional: start live streaming

Uncomment and run the cell below only when you want it to keep inserting new transactions.

In [ ]:
# stream(conn, interval=2.0)